In [13]:
!pip install transformers sentencepiece gradio nltk -q

In [14]:
import shutil
shutil.rmtree("/root/.cache/huggingface", ignore_errors=True)
print("Cache cleared")

Cache cleared


In [15]:
import shutil
shutil.rmtree("/root/.cache/huggingface", ignore_errors=True)
print("Cache cleared")

Cache cleared


In [19]:
model_name = "valhalla/t5-base-qg-hl"
print("Downloading tokenizer...")
tokenizer = T5Tokenizer.from_pretrained(model_name)
print("Downloading model... (this takes 2-3 mins, don't interrupt)")
model = T5ForConditionalGeneration.from_pretrained(model_name)
print(" Model loaded successfully!")

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

 Model loaded successfully!


In [20]:
def generate_questions_from_passage(passage):
    sentences = sent_tokenize(passage)
    questions = []
    for sentence in sentences:
        sentence = sentence.strip()
        word_count = len(sentence.split())
        if word_count < 5 or word_count > 50:
            continue
        input_text = f"{sentence} <hl> {sentence} <hl>"
        inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=64, num_beams=4, no_repeat_ngram_size=2, early_stopping=True)
        question = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if question.strip() == "" or "true" in question.lower() or "false" in question.lower():
            continue
        questions.append({"question": question, "answer": sentence})
    return questions

def gradio_fn(passage):
    if not passage.strip():
        return "Please enter a passage."
    results = generate_questions_from_passage(passage)
    if not results:
        return "Could not generate questions. Try a clearer passage."
    output = ""
    for i, r in enumerate(results, 1):
        output += f"Q{i}: {r['question']}\n"
        output += f"Answer: {r['answer']}\n"
        output += "-" * 50 + "\n"
    return output

gr.Interface(
    fn=gradio_fn,
    inputs=gr.Textbox(label="Paste Your Passage", lines=10, placeholder="Enter any paragraph..."),
    outputs=gr.Textbox(label="Generated Questions", lines=20),
    title=" Automatic Exam Question Generator",
    description="Paste any passage → get exam questions automatically"
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://77d22e6dabfd40d59b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
